[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/06_Functions/Functions_Deep_Dive.ipynb)

# 1.6 ONNX Functions — Deep Dive

ONNX Functions let you **define reusable operator combinations** — composing primitive operators into higher-level abstractions that behave like custom operators, without writing any C++ kernels.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Motivation: Why Functions?](#section-1) | The problem of repeated subgraph patterns |
| 2 | [Function Composition — Formal Definition](#section-2) | Mathematical framework for operator composition |
| 3 | [FunctionProto Structure](#section-3) | Anatomy of an ONNX function definition |
| 4 | [Building a Function Without Attributes](#section-4) | LinearRegression function example |
| 5 | [Building a Function With Attributes](#section-5) | Parameterized functions |
| 6 | [Using Functions in Models](#section-6) | Invoking custom functions |
| 7 | [Functions vs Subgraphs](#section-7) | When to use each |
| 8 | [Function Inlining and Optimization](#section-8) | How runtimes handle functions |
| 9 | [Key Takeaways & Interview Questions](#section-9) | Summary |

### Prerequisites

- Completed **1.1–1.5** (graph construction through control flow)
- Understanding of operator schemas and opsets

<a id='section-1'></a>
## Section 1: Motivation — Why Functions?

### The Repetition Problem

Many models contain **repeated subgraph patterns**. A transformer model, for example, contains the same attention pattern repeated $L$ times:

```
Without functions (flat graph):          With functions (structured):

┌────────────────────────┐              ┌────────────────────────┐
│ MatMul Q1              │              │ Attention(X, W_Q1...) │
│ MatMul K1              │              │     (function call)    │
│ MatMul V1              │              ├────────────────────────┤
│ Scale, Softmax, MatMul │              │ Attention(X, W_Q2...) │
│ Add, LayerNorm         │              │     (function call)    │
├────────────────────────┤              ├────────────────────────┤
│ MatMul Q2              │              │ Attention(X, W_Q3...) │
│ MatMul K2              │              │     (function call)    │
│ MatMul V2              │              └────────────────────────┘
│ Scale, Softmax, MatMul │
│ Add, LayerNorm         │              Function defined ONCE:
├────────────────────────┤              ┌────────────────────────┐
│ MatMul Q3              │              │ Attention(X, W_Q, ..):│
│ MatMul K3              │              │   Q = MatMul(X, W_Q)  │
│ ...                    │              │   K = MatMul(X, W_K)  │
└────────────────────────┘              │   V = MatMul(X, W_V)  │
                                        │   ...                 │
30+ nodes per layer × L layers          └────────────────────────┘
```

### Benefits of Functions

| Benefit | Description |
|---------|-------------|
| **Readability** | Shorter graphs with meaningful operator names |
| **Reusability** | Define once, use many times across the model |
| **Optimization** | Runtime can provide fused/optimized implementations |
| **Extensibility** | Create domain-specific operators without C++ kernels |
| **Portability** | Functions decompose to standard ops, ensuring any runtime can execute them |

<a id='section-2'></a>
## Section 2: Function Composition — Formal Definition

### Definition 2.1 (Function Composition)

An ONNX function defines a **composition** of standard operators into a new named operator:

$$g = f_n \circ f_{n-1} \circ \cdots \circ f_1$$

where each $f_i$ is a standard ONNX operator. The composite function $g$ can then be invoked as if it were a primitive operator.

### Example: Linear Regression as Composition

$$\text{LinearRegression}(X, A, B) = \text{Add}(\text{MatMul}(X, A), B)$$

This is the composition:

$$g = \text{Add} \circ (\text{MatMul} \times \text{id})$$

More precisely, using function notation:

$$g: \mathbb{R}^{N \times D} \times \mathbb{R}^{D \times K} \times \mathbb{R}^{K} \to \mathbb{R}^{N \times K}$$
$$g(X, A, B) = XA + B$$

### Definition 2.2 (ONNX FunctionProto)

Formally, a `FunctionProto` $F$ is defined by:

$$F = (\text{domain}, \text{name}, \text{inputs}, \text{outputs}, \text{nodes}, \text{opsets}, \text{attrs})$$

where:
- **domain**: Namespace for the function (e.g., `"custom"`)
- **name**: Function identifier (e.g., `"LinearRegression"`)
- **inputs**: Ordered list of input names
- **outputs**: Ordered list of output names
- **nodes**: The computation graph body (list of `NodeProto`)
- **opsets**: Which opsets the body uses
- **attrs**: Attribute names the function accepts (optional)

### The Function Hierarchy

```
ModelProto
├── graph: GraphProto
│   └── node[]: NodeProto
│       └── Some nodes reference functions:
│           op_type = "LinearRegression"
│           domain  = "custom"
│
└── functions[]: FunctionProto     ◀── Definitions live here
    └── FunctionProto
        ├── domain: "custom"
        ├── name: "LinearRegression"
        ├── input: ["X", "A", "B"]
        ├── output: ["Y"]
        ├── node[]: [MatMul, Add]  ◀── Body (standard ops)
        └── opset_import[]
```

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy matplotlib

In [ ]:
import numpy as np
from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid, make_function)
from onnx.checker import check_model
import onnxruntime as ort

print('Setup complete!')

<a id='section-3'></a>
## Section 3: FunctionProto Structure

### The `make_function()` API

```python
make_function(
    domain,           # str: namespace (e.g., "custom")
    fname,            # str: function name
    inputs,           # list[str]: input parameter names
    outputs,          # list[str]: output names
    nodes,            # list[NodeProto]: body computation
    opset_imports,    # list[OpsetIdProto]: which opsets the body uses
    attributes,       # list[str]: attribute names (empty if none)
)
```

### Key Design Decisions

1. **Functions are untyped**: Unlike `make_graph()` which takes `ValueInfoProto` (typed) inputs, `make_function()` takes plain string names. Type checking happens at the call site.

2. **Functions live on the ModelProto**: They are stored in `ModelProto.functions`, not inside the graph. This enables sharing across subgraphs.

3. **Functions require a domain**: Every function must belong to a custom domain (not the default `""` domain). This prevents conflicts with standard operators.

<a id='section-4'></a>
## Section 4: Building a Function Without Attributes

Let's define a `LinearRegression` function that computes $Y = XA + B$.

In [ ]:
# ── Step 1: Define the function ──────────────────────────────────

new_domain = 'custom'
opset_imports = [make_opsetid('', 14), make_opsetid(new_domain, 1)]

# Function body: two standard ONNX operators
fn_node1 = make_node('MatMul', ['X', 'A'], ['XA'])
fn_node2 = make_node('Add', ['XA', 'B'], ['Y'])

linear_regression = make_function(
    new_domain,            # domain
    'LinearRegression',    # function name
    ['X', 'A', 'B'],      # input names
    ['Y'],                 # output names
    [fn_node1, fn_node2],  # body nodes
    opset_imports,         # opsets used by body
    [])                    # attributes (none)

print('Function defined:')
print(f'  Domain:  {linear_regression.domain}')
print(f'  Name:    {linear_regression.name}')
print(f'  Inputs:  {list(linear_regression.input)}')
print(f'  Outputs: {list(linear_regression.output)}')
print(f'  Nodes:   {[(n.op_type, list(n.input), list(n.output)) for n in linear_regression.node]}')

In [ ]:
# ── Step 2: Use the function in a graph ──────────────────────────

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

# Invoke the function like a regular operator
call_node = make_node('LinearRegression', ['X', 'A', 'B'], ['Y1'],
                      domain=new_domain)

# Chain with another standard op
abs_node = make_node('Abs', ['Y1'], ['Y'])

graph = make_graph([call_node, abs_node], 'func_demo', [X, A, B], [Y])

model_fn = make_model(
    graph,
    opset_imports=opset_imports,
    functions=[linear_regression])

check_model(model_fn)

print('Model with custom function created!')
print(f'  Graph nodes: {[(n.op_type, n.domain) for n in model_fn.graph.node]}')
print(f'  Functions:   {[(f.name, f.domain) for f in model_fn.functions]}')

In [ ]:
# ── Step 3: Run inference ────────────────────────────────────────
sess = ort.InferenceSession(
    model_fn.SerializeToString(),
    providers=['CPUExecutionProvider'])

x = np.array([[1, 2], [3, 4]], dtype=np.float32)
a = np.array([[0.5, -0.3], [0.8, 0.2]], dtype=np.float32)
b = np.array([0.1, -0.5], dtype=np.float32)

result = sess.run(None, {'X': x, 'A': a, 'B': b})[0]
expected = np.abs(x @ a + b)

print(f'Input X:\n{x}')
print(f'\nONNX result (|XA + B|):\n{result}')
print(f'\nNumPy expected:\n{expected}')
print(f'\nMatch: {np.allclose(result, expected)}')

<a id='section-5'></a>
## Section 5: Building a Function With Attributes

### Parameterized Functions

Functions can accept **attributes** — compile-time parameters that modify their behavior. This uses the `ref_attr_name` mechanism to forward attributes from the call site to inner nodes.

### Example: Parameterized Activation

Let's create a function `LinearWithActivation` that applies a configurable activation:

$$g(X, A, B; \alpha) = \text{LeakyRelu}(XA + B; \alpha)$$

where $\alpha$ is the negative slope attribute of LeakyRelu.

### The `ref_attr_name` Mechanism

```
Call site:                              Function body:

Node(                                   LeakyRelu node:
  op_type="LinearWithAct",                attribute:
  domain="custom",                          name="alpha"
  attribute={alpha: 0.01}   ──────▶         ref_attr_name="alpha"
)                                           (resolved to 0.01)
```

In [ ]:
from onnx.helper import make_attribute

# Function body with attribute reference
fn_matmul = make_node('MatMul', ['X', 'A'], ['XA'])
fn_add = make_node('Add', ['XA', 'B'], ['pre_act'])

# LeakyRelu with attribute forwarded from function call
fn_act = make_node('LeakyRelu', ['pre_act'], ['Y'])
attr = make_attribute('alpha', 0.01)  # default
attr.ref_attr_name = 'alpha'  # forward from call site
fn_act.attribute.append(attr)

linear_with_act = make_function(
    'custom',
    'LinearWithActivation',
    ['X', 'A', 'B'],
    ['Y'],
    [fn_matmul, fn_add, fn_act],
    [make_opsetid('', 14), make_opsetid('custom', 1)],
    ['alpha'])  # declare 'alpha' as a function attribute

print('Function with attributes defined:')
print(f'  Name:       {linear_with_act.name}')
print(f'  Attributes: {list(linear_with_act.attribute)}')

# Use it in a model
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

call = make_node('LinearWithActivation', ['X', 'A', 'B'], ['Y'],
                 domain='custom', alpha=0.1)

graph = make_graph([call], 'attr_demo', [X, A, B], [Y])
model_attr = make_model(
    graph,
    opset_imports=[make_opsetid('', 14), make_opsetid('custom', 1)],
    functions=[linear_with_act])
check_model(model_attr)
print('Model with parameterized function created!')

<a id='section-6'></a>
## Section 6: Using Functions in Models

### Multiple Function Calls

A function can be called **multiple times** in the same graph, each time with different inputs. This is the primary reusability benefit.

### Example: Two-Stage Pipeline

Let's build a model that applies `LinearRegression` twice:

$$Z = \text{LR}_2(\text{LR}_1(X, A_1, B_1), A_2, B_2) = (XA_1 + B_1)A_2 + B_2$$

In [ ]:
# Use LinearRegression function twice (two-layer network)

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 4])
A1 = make_tensor_value_info('A1', TensorProto.FLOAT, [4, 3])
B1 = make_tensor_value_info('B1', TensorProto.FLOAT, [3])
A2 = make_tensor_value_info('A2', TensorProto.FLOAT, [3, 2])
B2 = make_tensor_value_info('B2', TensorProto.FLOAT, [2])
Z = make_tensor_value_info('Z', TensorProto.FLOAT, [None, 2])

# First call: H = LinearRegression(X, A1, B1)
call1 = make_node('LinearRegression', ['X', 'A1', 'B1'], ['H'],
                  domain='custom', name='layer1')

# Second call: Z = LinearRegression(H, A2, B2)
call2 = make_node('LinearRegression', ['H', 'A2', 'B2'], ['Z'],
                  domain='custom', name='layer2')

graph = make_graph([call1, call2], 'two_stage',
                   [X, A1, B1, A2, B2], [Z])

model_multi = make_model(
    graph,
    opset_imports=[make_opsetid('', 14), make_opsetid('custom', 1)],
    functions=[linear_regression])
check_model(model_multi)

# Run inference
sess = ort.InferenceSession(
    model_multi.SerializeToString(),
    providers=['CPUExecutionProvider'])

np.random.seed(42)
x = np.random.randn(5, 4).astype(np.float32)
a1 = np.random.randn(4, 3).astype(np.float32)
b1 = np.random.randn(3).astype(np.float32)
a2 = np.random.randn(3, 2).astype(np.float32)
b2 = np.random.randn(2).astype(np.float32)

result = sess.run(None, {'X': x, 'A1': a1, 'B1': b1, 'A2': a2, 'B2': b2})[0]
expected = (x @ a1 + b1) @ a2 + b2

print(f'Two-stage LinearRegression pipeline:')
print(f'  X shape:      {x.shape}')
print(f'  Output shape: {result.shape}')
print(f'  Match:        {np.allclose(result, expected, atol=1e-6)}')
print(f'\n  Same function called {len(model_multi.graph.node)} times')
print(f'  Function defined only once in model.functions')

<a id='section-7'></a>
## Section 7: Functions vs Subgraphs

### Comparison

| Feature | Functions | Subgraphs (If/Loop/Scan) |
|:--------|:----------|:------------------------|
| **Purpose** | Reusable operator combinations | Control flow |
| **Where defined** | `ModelProto.functions` | Node attributes (`then_branch`, `body`) |
| **Invoked by** | Regular node with function's domain | Control flow operators only |
| **Reusable** | Yes, across the entire model | No, tied to one specific node |
| **Typed inputs** | No (type-erased) | Yes (`ValueInfoProto`) |
| **Attributes** | Supported via `ref_attr_name` | Inherited from parent scope |
| **Runtime handling** | Inlined/expanded before execution | Executed dynamically |

### When to Use Each

```
Need to REPEAT a pattern?  ──▶ Use a Function
  (same ops, different data)

Need CONDITIONAL execution? ──▶ Use If (subgraph)
  (only run if condition met)

Need ITERATION?            ──▶ Use Loop/Scan (subgraph)
  (repeat with state)
```

### Key Insight: Functions are Syntactic Sugar

At execution time, runtimes typically **inline** (expand) function calls into the main graph. The function definition serves as a template:

$$\text{expand}(\text{Node}(\text{LinearRegression}, [X, A, B])) = [\text{Node}(\text{MatMul}, [X, A]), \text{Node}(\text{Add}, [\cdot, B])]$$

This means functions have **zero runtime overhead** — they are purely a graph organization tool.

<a id='section-8'></a>
## Section 8: Function Inlining and Optimization

### The Inlining Process

When a runtime encounters a function call, it:

1. **Looks up** the function definition in `ModelProto.functions`
2. **Substitutes** the function's formal parameters with actual arguments
3. **Renames** internal tensor names to avoid conflicts (alpha-renaming)
4. **Inserts** the body nodes into the main graph

```
Before inlining:                   After inlining:

Main Graph:                        Main Graph:
  X ──▶ LinearRegression ──▶ Y      X ──▶ MatMul ──▶ _lr_XA
                                         _lr_XA ──▶ Add ──▶ Y

Function def:                      (function def still exists
  MatMul(X,A) → XA                  but is no longer referenced)
  Add(XA,B) → Y
```

### Optimization Opportunities

After inlining, the runtime can apply standard graph optimizations:

- **Constant folding**: If function inputs are initializers, pre-compute results
- **Operator fusion**: Fuse function body operators with surrounding nodes
- **Dead code elimination**: Remove function outputs that are never consumed
- **Pattern matching**: Recognize the expanded pattern and use an optimized kernel

Some runtimes may also provide **native implementations** of well-known functions (e.g., a fused `LinearRegression` kernel), bypassing inlining entirely.

In [ ]:
# List all make_* helper functions available in onnx.helper
import onnx.helper

make_functions = sorted(k for k in dir(onnx.helper) if k.startswith('make'))

print('Available make_* helper functions:')
print('=' * 50)
for fn in make_functions:
    doc = getattr(onnx.helper, fn).__doc__
    first_line = (doc or '').strip().split('\n')[0][:60]
    print(f'  {fn:35s} {first_line}')

<a id='section-9'></a>
## Section 9: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **Functions** | Reusable compositions of standard operators, stored in `ModelProto.functions` |
| **Composition** | $g = f_n \circ \cdots \circ f_1$ where each $f_i$ is a standard ONNX op |
| **Domain** | Functions must belong to a custom domain (not `""`) |
| **Attributes** | Forwarded from call site via `ref_attr_name` mechanism |
| **Inlining** | Runtimes expand functions into the main graph before execution — zero overhead |
| **vs Subgraphs** | Functions are for reuse; subgraphs are for control flow |

### Interview Questions

1. **Q**: What is an ONNX function and how does it differ from a subgraph?
   - **A**: A function is a reusable composition of operators stored in `ModelProto.functions`, invoked like a regular node. A subgraph is an embedded graph used for control flow (If/Loop/Scan). Functions are syntactic sugar (inlined at runtime); subgraphs control execution flow.

2. **Q**: How do ONNX functions handle attributes?
   - **A**: Through `ref_attr_name`: inner nodes reference function-level attributes that are bound at the call site. This enables parameterized functions without fixing values in the definition.

3. **Q**: What is the mathematical formulation of function composition in ONNX?
   - **A**: $g = f_n \circ f_{n-1} \circ \cdots \circ f_1$ where each $f_i$ is a standard operator. The composite $g$ maps input tensors to output tensors through the ordered application of body operators.

4. **Q**: Do ONNX functions have runtime overhead?
   - **A**: No. Functions are inlined (expanded) into the main graph before execution. After inlining, standard graph optimizations apply. Some runtimes may even provide fused kernels for recognized function patterns.

---

**Next:** [Parsing and Checker](../07_Parsing_and_Checker/) — Concise text format and model validation.